# PII-BR — treino do modelo NER

Fine-tuning do BERTimbau base para detectar 9 tipos de PII em português do Brasil, com o dataset [PII-BR](https://huggingface.co/datasets).

**Como rodar no Kaggle** (GPU grátis):
1. *Settings → Accelerator → GPU T4* e *Settings → Internet → On*.
2. *Add-ons → Secrets*: crie `HF_TOKEN` com um token de escrita do Hugging Face e marque-o para este notebook.
3. Na célula de configuração, troque `SEU_USUARIO` pelo seu usuário do Hugging Face.
4. *Save Version → Save & Run All* (roda em segundo plano, ~30 min).

No Colab: *Ambiente de execução → T4 GPU* e o mesmo secret `HF_TOKEN` (ícone de chave). Localmente, `SMOKE=1` roda uma versão mínima em CPU para testar o código.

In [ ]:
import os
import sys

# ---- Configuração -----------------------------------------------------------------
HF_USER = os.environ.get("HF_USER", "SEU_USUARIO")  # <- troque pelo seu usuário do HF
DATASET_REPO = os.environ.get("DATASET_REPO", f"{HF_USER}/PII-BR")
HUB_MODEL_ID = os.environ.get("HUB_MODEL_ID", f"{HF_USER}/PII-BR")
BASE_MODEL = os.environ.get("BASE_MODEL", "neuralmind/bert-base-portuguese-cased")

SMOKE = os.environ.get("SMOKE") == "1"  # teste rápido em CPU: 200 frases, 1 época, sem push
LOCAL_DATA = os.environ.get("LOCAL_DATA")  # pasta com train.jsonl etc. (em vez do Hub)
TRAIN_LIMIT = int(os.environ.get("TRAIN_LIMIT", "200" if SMOKE else "0"))  # 0 = tudo
PUSH = os.environ.get("PUSH", "0" if SMOKE else "1") == "1"

ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
OUTPUT_DIR = os.environ.get(
    "OUTPUT_DIR", "/kaggle/working/pii-br" if ON_KAGGLE else "outputs/pii-br"
)

SEED = 42
LEARNING_RATE = 5e-5
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "4"))
BATCH = 16
MAX_LENGTH = 256
WARMUP = 0.1  # fração dos passos (transformers 5: warmup_steps aceita fração)
WEIGHT_DECAY = 0.01

# Mesma ordem de pii_br.types.BIO_LABELS (um teste do repositório garante isso).
LABELS = [
    "O",
    "B-NOME",
    "I-NOME",
    "B-CPF",
    "I-CPF",
    "B-CNPJ",
    "I-CNPJ",
    "B-RG",
    "I-RG",
    "B-TELEFONE",
    "I-TELEFONE",
    "B-EMAIL",
    "I-EMAIL",
    "B-ENDERECO",
    "I-ENDERECO",
    "B-CEP",
    "I-CEP",
    "B-DATA_NASC",
    "I-DATA_NASC",
]
print(f"smoke={SMOKE} kaggle={ON_KAGGLE} colab={ON_COLAB} push={PUSH} -> {OUTPUT_DIR}")

In [ ]:
# Versões testadas no repositório. No Kaggle/Colab instala por cima das da imagem.
import subprocess

if ON_KAGGLE or ON_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers==5.23.0",
            "datasets>=4,<6",
            "accelerate>=1.10",
            "seqeval==1.2.2",
        ],
        check=True,
    )

In [ ]:
# Token do Hugging Face: só é preciso para publicar o modelo.
def get_hf_token():
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF_TOKEN")
    if ON_COLAB:
        from google.colab import userdata

        return userdata.get("HF_TOKEN")
    return os.environ.get("HF_TOKEN")


HF_TOKEN = get_hf_token() if PUSH else None
if PUSH and not HF_TOKEN:
    raise RuntimeError("PUSH ligado, mas HF_TOKEN não foi encontrado nos secrets.")

In [ ]:
from datasets import DatasetDict, load_dataset

SPLITS = ["train", "validation", "test", "test_hard", "test_real"]
if LOCAL_DATA:
    files = {
        s: f"{LOCAL_DATA}/{s}.jsonl" for s in SPLITS if os.path.exists(f"{LOCAL_DATA}/{s}.jsonl")
    }
    raw = load_dataset("json", data_files=files)
else:
    raw = load_dataset(DATASET_REPO)

if TRAIN_LIMIT:
    eval_limit = max(60, TRAIN_LIMIT // 10)
    raw = DatasetDict(
        {
            name: ds.shuffle(seed=SEED).select(
                range(min(len(ds), TRAIN_LIMIT if name == "train" else eval_limit))
            )
            for name, ds in raw.items()
        }
    )
print(raw)

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)


def tokenize_and_align(batch):
    """Rótulo no primeiro subtoken de cada palavra; os demais recebem -100."""
    enc = tokenizer(
        batch["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LENGTH
    )
    enc["labels"] = []
    for i, tags in enumerate(batch["ner_tags"]):
        previous, labels = None, []
        for word in enc.word_ids(i):
            labels.append(-100 if word is None or word == previous else tags[word])
            previous = word
        enc["labels"].append(labels)
    return enc


tokenized = raw.map(tokenize_and_align, batched=True, remove_columns=raw["train"].column_names)

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from seqeval.scheme import IOB2


def to_tag_lists(predictions, label_ids):
    preds = np.argmax(predictions, axis=-1)
    true_tags, pred_tags = [], []
    for p_row, l_row in zip(preds, label_ids):
        keep = l_row != -100
        true_tags.append([LABELS[i] for i in l_row[keep]])
        pred_tags.append([LABELS[i] for i in p_row[keep]])
    return true_tags, pred_tags


def compute_metrics(eval_pred):
    true_tags, pred_tags = to_tag_lists(eval_pred.predictions, eval_pred.label_ids)
    kwargs = {"mode": "strict", "scheme": IOB2, "zero_division": 0}
    return {
        "precision": precision_score(true_tags, pred_tags, **kwargs),
        "recall": recall_score(true_tags, pred_tags, **kwargs),
        "f1": f1_score(true_tags, pred_tags, **kwargs),
    }

In [ ]:
import torch
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

set_seed(SEED)
model = AutoModelForTokenClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)),
    label2id={label: i for i, label in enumerate(LABELS)},
)
cuda = torch.cuda.is_available()
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    warmup_steps=WARMUP,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=cuda and "deberta" not in BASE_MODEL,  # DeBERTa-v3 diverge em fp16
    train_sampling_strategy="group_by_length",
    logging_steps=50,
    report_to="none",
    seed=SEED,
    use_cpu=not cuda,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
# Retoma do último checkpoint se a sessão anterior caiu no meio.
last = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
print("retomando de", last) if last else None
trainer.train(resume_from_checkpoint=last)

In [ ]:
import json

results = {
    "base_model": BASE_MODEL,
    "dataset": LOCAL_DATA or DATASET_REPO,
    "epochs": EPOCHS,
    "splits": {},
}
for split in ["test", "test_hard", "test_real"]:
    if split not in tokenized:
        continue
    out = trainer.predict(tokenized[split], metric_key_prefix=split)
    true_tags, pred_tags = to_tag_lists(out.predictions, out.label_ids)
    report = classification_report(
        true_tags, pred_tags, mode="strict", scheme=IOB2, output_dict=True, zero_division=0
    )
    results["splits"][split] = {k: {m: float(v) for m, v in d.items()} for k, d in report.items()}
    print(f"===== {split} =====")
    print(
        classification_report(
            true_tags, pred_tags, mode="strict", scheme=IOB2, digits=3, zero_division=0
        )
    )

trainer.save_model(OUTPUT_DIR)
with open(f"{OUTPUT_DIR}/metrics_seqeval.json", "w") as f:
    json.dump(results, f, indent=2)

In [ ]:
if PUSH:
    from huggingface_hub import HfApi

    trainer.model.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN, commit_message="Treino PII-BR")
    tokenizer.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
    HfApi(token=HF_TOKEN).upload_file(
        path_or_fileobj=f"{OUTPUT_DIR}/metrics_seqeval.json",
        path_in_repo="metrics_seqeval.json",
        repo_id=HUB_MODEL_ID,
    )
    print(f"Modelo publicado: https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("PUSH desligado: modelo salvo só em", OUTPUT_DIR)